In [ ]:
import json
import math
import numpy as np
import os
import pandas as pd
import scipy.stats as stats

from IPython.display import display
from collections import defaultdict
from sklearn import metrics

import matplotlib as mpl
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = 'retina'

### Utility functions

In [ ]:
def merge_data_diy(df_pairs, df_similarity):
    df_pairs = df_pairs.merge(
        df_similarity,
        how='left',
        left_on=['idb_path_1', 'fva_1', 'idb_path_2', 'fva_2'],
        right_on=['idb_path_1', 'fva_1', 'idb_path_2', 'fva_2'])
    
    return df_pairs

In [ ]:
def compute_ranking(df_pos, df_neg, test_name, r_dict, rank_method):
    TH_LIST = [1] + list(range(5, 55, 5))
    # print("df_pos_testing.columns:", df_pos.columns)    
    # print("df_pos:", df_pos)    
    for task in sorted(set(df_pos['db_type'])):
        df_pos_task = df_pos[df_pos['db_type'] == task]
        df_neg_task = df_neg[df_neg['db_type'] == task]

        # print(df_neg_task.groupby(['idb_path_1', 'fva_1']).size().value_counts())

        tot_pos = df_pos_task.shape[0]

        # Compute the ranking for all the positive test cases
        rank_list = list()
        for idx, group in df_neg_task.groupby(['idb_path_1', 'fva_1']):
            c1 = (df_pos_task['idb_path_1'] == idx[0])
            c2 = (df_pos_task['fva_1'] == idx[1])
            pos_pred = df_pos_task[c1 & c2]['sim'].values[0]
            neg_pred = list(group['sim'].values)
            # print(len(neg_pred))
            # assert(len(neg_pred) == 100)
            ranks = stats.rankdata([pos_pred] + neg_pred, method=rank_method)
            rank_list.append(102 - ranks[0])

        # Compute the ranking list
        cc_list = list()
        for th in TH_LIST:
            cc_list.append(len([x for x in rank_list if x <= th]))

        # MRR@10 metric
        tmp_list = [1 / x if x <= 10 else 0 for x in rank_list]
        MRR = sum(tmp_list) / len(tmp_list)

        # Save data in a temporary dictionary
        if task not in r_dict:
            r_dict[task] = defaultdict(list)

        r_dict[task]['Model Name'].append(test_name)
        for th, cc in zip(TH_LIST, cc_list):
            r_dict[task]["Recall@{}".format(th)].append(cc / tot_pos)
        r_dict[task]["MRR@10"].append(MRR)

In [ ]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

def from_dict_to_df_and_plot(r_dict, output_dir, rank_method):
    # ─────────────────────────────────────────────────────────────────────
    # 0) GLOBAL FONT CONFIGURATION (once):
    mpl.rcParams['font.family']      = 'sans-serif'
    mpl.rcParams['font.sans-serif']  = ['Helvetica', 'Arial', 'DejaVu Sans']
    mpl.rcParams['font.size']        = 16
    mpl.rcParams['axes.titlesize']   = 14
    mpl.rcParams['axes.labelsize']   = 16
    mpl.rcParams['legend.fontsize']  = 14
    mpl.rcParams['xtick.labelsize']  = 14
    mpl.rcParams['ytick.labelsize']  = 14

    # ─────────────────────────────────────────────────────────────────────
    # 1) Build a pool of valid marker strings by testing each key
    markers_pool = []
    for marker_key in mpl.markers.MarkerStyle.markers:
        try:
            mpl.markers.MarkerStyle(marker_key)
            if marker_key and not marker_key.lower().startswith(('tick', 'caret')):
                markers_pool.append(marker_key)
        except Exception:
            continue

    if not markers_pool:
        markers_pool = ['o', 's', '^', 'v', 'D', '*', '+', 'x']

    # ─────────────────────────────────────────────────────────────────────
    for task in sorted(r_dict.keys()):
        print(f"[D] Task: {task}")

        # 2) Build DataFrame and set index
        df_rank = pd.DataFrame.from_dict(r_dict[task])
        df_rank.set_index('Model Name', inplace=True)

        # 3) Save CSV for reference
        csv_name = f"df_MRR@10_Recall@K_{task}_{rank_method}.csv"
        df_rank.to_csv(os.path.join(output_dir, csv_name), index=True)

        # 4) Keep only Recall@5,10,…,50 columns, then transpose
        cols = [f"Recall@{k}" for k in range(5, 55, 5)]
        df_rank = df_rank[cols]
        df_rank.columns = range(5, 55, 5)
        df_rank = df_rank.transpose()

        # 5) Remove “blacklisted” models
        blacklist = [
            'GNN-s2v_GeminiNN_NoFeatures_e5',
            'GNN-s2v_GeminiNN_OPC-200_e5',
            'GNN-s2v_AttentionMean_e5',
            'GNN-s2v_RNN_ASM_e7',
            'catalog1_16', 'catalog1_32', 'catalog1_64',
            'GGSNN_NoFeatures_e10', 'GMN_NoFeatures_e16',
            'IMM:0.00_MNEM:0.00_GRAPH:1.00',
            'IMM:0.00_MNEM:1.00_GRAPH:1.00',
            'IMM:1.00_MNEM:1.00_GRAPH:1.00',
            'SAFE_ASM-list_250_e5',
            'SAFE_ASM-list_Trainable_e10',
            'SAFE_ASM-list_Rand_Trainable_e10',
            'pvdm_e10', 'pvdbow_e10'
        ]
        for bad in blacklist:
            if bad in df_rank.columns:
                del df_rank[bad]

        if df_rank.shape[1] == 0:
            print(f"    ↳ No models remain after filtering for '{task}'. Skipping.")
            continue

        # ─────────────────────────────────────────────────────────────────
        # 6) Create a rectangular figure (12×6) with the chosen font settings
        fig, ax = plt.subplots(figsize=(12, 6))

        # Force x‐ticks to [5, 10, 15, …, 50]
        x_values = df_rank.index.values
        ax.set_xticks(x_values)
        ax.set_xticklabels([str(x) for x in x_values])

        # Draw a light grid
        ax.grid(True, linestyle='--', alpha=0.4)

        # Plot each model’s Recall@K curve
        for idx, model in enumerate(df_rank.columns):
            marker = markers_pool[idx % len(markers_pool)]
            ax.plot(
                x_values,
                df_rank[model].values,
                label=model,
                marker=marker,
                markersize=6,
                linewidth=1.8,
                alpha=0.9
            )

        # 7) Labels (no title)
        ax.set_xlabel("K (number of returned results)")
        ax.set_ylabel("Recall@K")

        # ─────────────────────────────────────────────────────────────────
        # 8) Adjust y‐axis bounds to nearest 0.1 below min and nearest 0.1 above max
        data_min = df_rank.values.min()
        data_max = df_rank.values.max()

        y_lower = math.floor(data_min * 10) / 10
        y_upper = math.ceil(data_max * 10) / 10
        ax.set_ylim(y_lower, y_upper)

        # ─────────────────────────────────────────────────────────────────
        # 9) Only show legend for “XM”, placed inside at bottom right
        if task == "XC+XB" or task == "XOb":
            legend = ax.legend(
                loc='lower right',  # inside bottom-right corner
                frameon=True,
                borderpad=0.5
            )
            legend.get_frame().set_edgecolor('lightgray')
            legend.get_frame().set_linewidth(0.5)

        # 10) Tidy margins & save as PDF
        plt.tight_layout()
        out_path = os.path.join(output_dir, f"Recall@K_{task}_{rank_method}.pdf")
        fig.savefig(
            out_path,
            format='pdf',
            dpi=300,
            bbox_inches='tight',
            pad_inches=0.05
        )
        plt.close(fig)


In [ ]:
def compute_mrr_and_recall(df_pos, df_neg, results_dir, output_dir):
    # Alternatives: min or max
    rank_method = 'max'
    print("[D] Using rank_method: {}".format(rank_method))
    results_dict = dict()
    for csv_file in sorted(os.listdir(results_dir)):
        if (not csv_file.endswith(".csv")) or \
                ("pos_rank_testing" not in csv_file):
            continue

        print("[D] Processing\n\t{}\n\t{}".format(
            csv_file, csv_file.replace("pos_rank_testing", "neg_rank_testing")))
        

        df_pos_sim = pd.read_csv(
            os.path.join(results_dir, csv_file))

        df_neg_sim = pd.read_csv(
            os.path.join(results_dir, csv_file.replace(
                "pos_rank_testing", "neg_rank_testing")))

        assert(df_pos_sim.isna().sum()['sim'] == 0)
        assert(df_neg_sim.isna().sum()['sim'] == 0)

        # df_pos_m = merge_data(df_pos, df_pos_sim)
        # df_neg_m = merge_data(df_neg, df_neg_sim)
        
        # df_pos_m = df_pos_sim
        # df_neg_m = df_neg_sim

        if 'db_type' in df_pos_sim.columns:
            df_pos_m = df_pos_sim
        else:
            df_pos_m = df_pos_sim.copy()
            df_pos_m['db_type'] = df_pos['db_type'].values

        if 'db_type' in df_neg_sim.columns:
            df_neg_m = df_neg_sim
        else:
            df_neg_m = df_neg_sim.copy()
            df_neg_m['db_type'] = df_neg['db_type'].values  
        # df_pos_m = merge_data(df_pos, df_pos_sim)

        # # Check if db_type_x and db_type_y are the same for all rows
        # if (df_pos_m['db_type_x'] == df_pos_m['db_type_y']).all():
        #     df_pos_m = df_pos_m.drop(columns=['db_type_y'])
        #     df_pos_m = df_pos_m.rename(columns={'db_type_x': 'db_type'})
        # else:
        #     raise ValueError("Mismatch detected between 'db_type_x' and 'db_type_y'")
 

        # df_neg_m = merge_data(df_neg, df_neg_sim)

        # # Check if db_type_x and db_type_y are the same for all rows
        # if (df_neg_m['db_type_x'] == df_neg_m['db_type_y']).all():
        #     df_neg_m = df_neg_m.drop(columns=['db_type_y'])
        #     df_neg_m = df_neg_m.rename(columns={'db_type_x': 'db_type'})
        # else:
        #     raise ValueError("Mismatch detected between 'db_type_x' and 'db_type_y'")

        test_name = csv_file.replace("pos_rank_testing_", "")
        test_name = test_name.replace(".trex_out", "Trex")
        test_name = test_name.replace("IMM:4.00_MNEM:0.05_GRAPH:1.00", "FunctionSimSearch")
        test_name = test_name.replace("Massarelli", "Massarelli el al.")
        test_name = test_name.replace("asm2vec", "Asm2Vec")
        test_name = test_name.replace("catalog1_128", "FCatalog")
        test_name = test_name.replace("hermessim", "HermesSim")
        test_name = test_name.replace("Dataset-1_", "")
        test_name = test_name.replace("Dataset-1", "")
        test_name = test_name.replace("Dataset-2-CodeCMR", "")
        test_name = test_name.replace("Dataset-2_", "")
        test_name = test_name.replace("Dataset-3_", "")
        test_name = test_name.replace("Dataset-4_", "")
        test_name = test_name.replace("Dataset-5_", "")
        test_name = test_name.replace("Dataset-6_", "")
        test_name = test_name.replace("Dataset-2", "")
        test_name = test_name.replace("Dataset-3", "")
        test_name = test_name.replace("Dataset-4", "")
        test_name = test_name.replace("Dataset-5", "")
        test_name = test_name.replace("Dataset-6", "")
        test_name = test_name.replace("Dataset-7", "")
        # test_name = test_name.replace("_", "")
        test_name = test_name.replace(".csv", "")
        compute_ranking(df_pos_m, df_neg_m, test_name,
                        results_dict, rank_method=rank_method)

    from_dict_to_df_and_plot(results_dict, output_dir, rank_method)

In [ ]:
RESULTS_DIR = "../data/Dataset-1/"
OUTPUT_DIR = "metrics_and_plots/Dataset-1/"

base_path = "../../DBs/Dataset-1/pairs/testing/"

df_pos_testing = pd.read_csv(
    os.path.join(base_path, "pos_rank_testing_Dataset-1.csv"),
    index_col=0)

df_neg_testing = pd.read_csv(
    os.path.join(base_path, "neg_rank_testing_Dataset-1.csv"),
    index_col=0)

compute_mrr_and_recall(df_pos_testing, df_neg_testing, RESULTS_DIR, OUTPUT_DIR)

## Dataset 2

In [ ]:
RESULTS_DIR = "../data/Dataset-2/"
OUTPUT_DIR = "metrics_and_plots/Dataset-2/"

base_path = "../../DBs/Dataset-2/pairs/testing/"

df_pos_testing = pd.read_csv(
    os.path.join(base_path, "pos_rank_testing_Dataset-2.csv"),
    index_col=0)

df_neg_testing = pd.read_csv(
    os.path.join(base_path, "neg_rank_testing_Dataset-2.csv"),
    index_col=0)

compute_mrr_and_recall(df_pos_testing, df_neg_testing, RESULTS_DIR, OUTPUT_DIR)

In [ ]:
RESULTS_DIR = "../data/Dataset-3/"
OUTPUT_DIR = "metrics_and_plots/Dataset-3/"

base_path = "../../DBs/Dataset-3/pairs/testing/"

df_pos_testing = pd.read_csv(
    os.path.join(base_path, "pos_rank_testing_Dataset-3.csv"),
    index_col=0)

df_neg_testing = pd.read_csv(
    os.path.join(base_path, "neg_rank_testing_Dataset-3.csv"),
    index_col=0)

compute_mrr_and_recall(df_pos_testing, df_neg_testing, RESULTS_DIR, OUTPUT_DIR)

In [ ]:
RESULTS_DIR = "../data/Dataset-4/"
OUTPUT_DIR = "metrics_and_plots/Dataset-4/"

base_path = "../../DBs/Dataset-4/pairs/testing/"

df_pos_testing = pd.read_csv(
    os.path.join(base_path, "pos_rank_testing_Dataset-4.csv"),
    index_col=0)

df_neg_testing = pd.read_csv(
    os.path.join(base_path, "neg_rank_testing_Dataset-4.csv"),
    index_col=0)

compute_mrr_and_recall(df_pos_testing, df_neg_testing, RESULTS_DIR, OUTPUT_DIR)

In [ ]:
RESULTS_DIR = "../data/Dataset-5/"
OUTPUT_DIR = "metrics_and_plots/Dataset-5/"

base_path = "../../DBs/Dataset-5/pairs/testing/"

df_pos_testing = pd.read_csv(
    os.path.join(base_path, "pos_rank_testing_Dataset-5.csv"),
    index_col=0)

df_neg_testing = pd.read_csv(
    os.path.join(base_path, "neg_rank_testing_Dataset-5.csv"),
    index_col=0)

compute_mrr_and_recall(df_pos_testing, df_neg_testing, RESULTS_DIR, OUTPUT_DIR)

In [ ]:
RESULTS_DIR = "../data/Dataset-7/"
OUTPUT_DIR = "metrics_and_plots/Dataset-7/"

base_path = "../../DBs/Dataset-7/pairs/testing/"

df_pos_testing = pd.read_csv(
    os.path.join(base_path, "pos_rank_testing_Dataset-7.csv"),
    index_col=0)

df_neg_testing = pd.read_csv(
    os.path.join(base_path, "neg_rank_testing_Dataset-7.csv"),
    index_col=0)

compute_mrr_and_recall(df_pos_testing, df_neg_testing, RESULTS_DIR, OUTPUT_DIR)